## ModelOpt Q/DQ ONNX Export (All Seeds)

Export base ONNX from each trained checkpoint (seed_1, seed_2, seed_42), then quantize each with ModelOpt in INT8, FP8, and INT4.

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

SKIP_EXISTING = True

In [15]:
import numpy as np
import onnx
from onnx import TensorProto
import modelopt.onnx.quantization as moq

from src.config import ExperimentConfig
from src.data import build_runner_loaders
from src.model import get_model
from utils.onnx_exporter import ONNXExporter

## Export base ONNX

In [16]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
ONNX_DIR  = PROJECT_ROOT / "onnx"
INPUT_BITS = [8, 4, 2, 1]

for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        print(f"SKIP (no dir): {ckpt_dir}")
        continue

    seeds = sorted(
        int(d.name.split("_")[1])
        for d in ckpt_dir.iterdir()
        if d.is_dir() and (d / "best.pth").exists()
    )
    print(f"{bits}-bit seeds found: {seeds}")

    onnx_subdir = ONNX_DIR / f"fp32_{bits}bit"
    onnx_subdir.mkdir(parents=True, exist_ok=True)

    for seed in seeds:
        ckpt = str(ckpt_dir / f"seed_{seed}" / "best.pth")
        onnx_path = onnx_subdir / f"resnet_{seed}.onnx"

        if SKIP_EXISTING and onnx_path.exists():
            print(f"  SKIP (exists): {onnx_path}")
            continue

        model = get_model(cfg=None, pretrained=False, checkpoint_path=ckpt)
        exporter = ONNXExporter(model=model, device="cpu", onnx_path=str(onnx_path))
        exporter.export_model(opset_version=18, dynamic_batch=True)


8-bit seeds found: [1, 2, 42]
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_1.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_2.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_42.onnx
4-bit seeds found: [1, 2, 42]
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_2.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_42.onnx
2-bit seeds found: [1, 2, 42]
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_1.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_2.onnx
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_42.onnx
1-bit seeds found: [1, 2, 42]
  SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/r

## Quantization config

In [17]:
QUANT_DTYPES   = ["int8", "fp8", "int4"]
CALIB_BATCHES  = 32

## Calibration data

In [18]:
calib_data_by_bits = {}

for bits in INPUT_BITS:
    cfg = ExperimentConfig(device="cpu", batch_size=32, input_quant_bits=bits)
    loader, _ = build_runner_loaders(cfg)

    batches = []
    for i, (images, _) in enumerate(loader):
        if i >= CALIB_BATCHES:
            break
        batches.append(images.numpy())

    calib_data_by_bits[bits] = np.concatenate(batches, axis=0)
    print(f"  {bits}-bit calibration data: {calib_data_by_bits[bits].shape}")

print(f"\nCalibration data built for bit-depths: {list(calib_data_by_bits.keys())}")

[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
  8-bit calibration data: (1024, 3, 224, 224)
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
  4-bit calibration data: (1024, 3, 224, 224)
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
  2-bit calibration data: (1024, 3, 224, 224)
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
  1-bit calibration data: (1024, 3, 224, 224)

Calibration data built for bit-depths: [8, 4, 2, 1]


## Quantize with ModelOpt

In [19]:
_FP8_OPS = {"Conv", "Gemm", "MatMul", "Add"}

def quantize_for_bits(bits):
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        print(f"SKIP (no dir): {ckpt_dir}")
        return

    seeds = sorted(
        int(d.name.split("_")[1])
        for d in ckpt_dir.iterdir()
        if d.is_dir() and (d / "best.pth").exists()
    )

    onnx_subdir = ONNX_DIR / f"fp32_{bits}bit"
    calib_data = calib_data_by_bits[bits]

    for seed in seeds:
        onnx_base = str(onnx_subdir / f"resnet_{seed}.onnx")

        for dtype in QUANT_DTYPES:
            onnx_out = str(onnx_subdir / f"resnet_{seed}_{dtype}_qdq.onnx")

            if SKIP_EXISTING and Path(onnx_out).exists():
                print(f"SKIP (exists): {onnx_out}")
                continue

            print(f"\n{'='*60}")
            print(f"Quantizing bits={bits} seed={seed} dtype={dtype}", flush=True)
            print(f"  base: {onnx_base}")
            print(f"  out:  {onnx_out}")
            print(f"  calib: {bits}-bit input data ({calib_data.shape[0]} images)")

            nodes_to_q = None
            if dtype == "fp8":
                m = onnx.load(onnx_base)
                nodes_to_q = [n.name for n in m.graph.node if n.op_type in _FP8_OPS]
                print(f"  Explicit nodes_to_quantize: {len(nodes_to_q)} nodes")

            moq.quantize(
                onnx_path=onnx_base,
                quantize_mode=dtype,
                calibration_data=calib_data,
                output_path=onnx_out,
                op_types_to_quantize=list(_FP8_OPS) if dtype == "fp8" else None,
                nodes_to_quantize=nodes_to_q,
            )
            print(f"  Saved → {onnx_out}")

In [20]:
quantize_for_bits(8)

SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_1_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_1_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_1_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_2_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_2_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_2_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_42_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_42_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_8bit/resnet_42_int4_qdq.onnx


In [21]:
quantize_for_bits(4)


Quantizing bits=4 seed=1 dtype=int8
  base: /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1.onnx
  out:  /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1_int8_qdq.onnx
  calib: 4-bit input data (1024 images)
[modelopt][onnx] - INFO - Starting quantization process for model: /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1.onnx
[modelopt][onnx] - INFO - Quantization mode: int8
[modelopt][onnx] - INFO - Preprocessing the model /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1.onnx
[modelopt][onnx] - INFO - Model has dynamic inputs: ['images']
[modelopt][onnx] - INFO - Found 0 custom layers and 59 tensors
[modelopt][onnx] - INFO - No custom ops found. If that's not correct, please make sure that the 'tensorrt' python package is correctly installed and that the paths to 'libcudnn*.so' and TensorRT 'lib/' are in 'LD_LIBRARY_PATH'. If the custom op is not directly available as a plugin in TensorRT, please also make sure

100%|██████████| 46/46 [00:00<00:00, 1539.49it/s]


100%|██████████| 46/46 [00:00<00:00, 1586.26it/s]


100%|██████████| 46/46 [00:00<00:00, 1754.22it/s]


100%|██████████| 46/46 [00:00<00:00, 1825.39it/s]


...

Running clip search...: 100%|██████████| 1/1 [00:05<00:00,  5.61s/it]

[modelopt][onnx] - INFO - Clip search for all weights took 5.611982107162476 seconds



Quantizing the weights...: 100%|██████████| 1/1 [00:00<00:00, 391.44it/s]

[modelopt][onnx] - INFO - Quantizing actual weights took 0.004228830337524414 seconds
[modelopt][onnx] - INFO - Inserting DQ nodes took 0.00043845176696777344 seconds
[modelopt][onnx] - INFO - Exporting the quantized graph
[modelopt][onnx] - INFO - Loading extension modelopt_round_and_pack_ext...


[modelopt][onnx] - WARNING - copy_file() got an unexpected keyword argument 'dry_run'
Unable to load `modelopt_round_and_pack_ext', falling back to python based optimized version
[modelopt][onnx] - INFO - Exporting took 4.498872995376587 seconds
[modelopt][onnx] - INFO - INT4 Quantization completed in 10.44 seconds
[modelopt][onnx] - INFO - Removing 0 redundant Cast nodes
[modelopt][onnx] - INFO - Total number of nodes: 52
[modelopt][onnx] - INFO - Total number of quantized nodes: 1
[modelopt][onnx] - INFO - Quantized onnx model is saved as /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1_int4_qdq.onnx
[modelopt][onnx] - INFO - Cleaning up intermediate files
[modelopt][onnx] - INFO - Validating quantized model
[modelopt][onnx] - WARNING - ONNX model checker failed, check your deployment status
[modelopt][onnx] - WARNING - Unrecognized attribute: block_size for operator DequantizeLinear

==> Context: Bad node spec for node. Name: fc.weight_DequantizeLinear OpType: Dequ

In [22]:
quantize_for_bits(2)

SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_1_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_1_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_1_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_2_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_2_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_2_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_42_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_42_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_2bit/resnet_42_int4_qdq.onnx


In [23]:
quantize_for_bits(1)

SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_1_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_1_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_1_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_2_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_2_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_2_int4_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_42_int8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_42_fp8_qdq.onnx
SKIP (exists): /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_1bit/resnet_42_int4_qdq.onnx


## Q/DQ counts

In [24]:
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    seeds = sorted(
        int(d.name.split("_")[1])
        for d in ckpt_dir.iterdir()
        if d.is_dir() and (d / "best.pth").exists()
    )
    onnx_subdir = ONNX_DIR / f"fp32_{bits}bit"
    for seed in seeds:
        print(f"\n--- {bits}bit / seed {seed} ---")
        for dtype_label in QUANT_DTYPES:
            path = onnx_subdir / f"resnet_{seed}_{dtype_label}_qdq.onnx"
            if not path.exists():
                print(f"  {dtype_label:>5}: not exported yet")
                continue
            ops = [n.op_type for n in onnx.load(str(path)).graph.node]
            print(f"  {dtype_label:>5}: Q={ops.count('QuantizeLinear'):3d}  DQ={ops.count('DequantizeLinear'):3d}")



--- 8bit / seed 1 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 8bit / seed 2 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 8bit / seed 42 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 4bit / seed 1 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 4bit / seed 2 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 4bit / seed 42 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 2bit / seed 1 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 2bit / seed 2 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 2bit / seed 42 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 1bit / seed 1 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4: Q=  0  DQ=  1

--- 1bit / seed 2 ---
   int8: Q= 41  DQ= 41
    fp8: Q= 46  DQ= 46
   int4:

## Initializer dtypes

In [25]:
dtype_map = {v: k for k, v in TensorProto.DataType.items()}

# To visualize any ONNX graph, drag-drop the file into https://netron.app

# --- uncomment one ---
# path = ONNX_DIR / "fp32_8bit" / "resnet_42_int8_qdq.onnx"
# path = ONNX_DIR / "fp32_8bit" / "resnet_42_fp8_qdq.onnx"
# path = ONNX_DIR / "fp32_8bit" / "resnet_42_int4_qdq.onnx"
# path = ONNX_DIR / "fp32_4bit" / "resnet_42_int8_qdq.onnx"
# path = ONNX_DIR / "fp32_4bit" / "resnet_42_fp8_qdq.onnx"
# path = ONNX_DIR / "fp32_4bit" / "resnet_42_int4_qdq.onnx"
# path = ONNX_DIR / "fp32_2bit" / "resnet_42_int8_qdq.onnx"
# path = ONNX_DIR / "fp32_2bit" / "resnet_42_fp8_qdq.onnx"
# path = ONNX_DIR / "fp32_2bit" / "resnet_42_int4_qdq.onnx"
# path = ONNX_DIR / "fp32_1bit" / "resnet_42_int8_qdq.onnx"
# path = ONNX_DIR / "fp32_1bit" / "resnet_42_fp8_qdq.onnx"
# path = ONNX_DIR / "fp32_1bit" / "resnet_42_int4_qdq.onnx"

m = onnx.load(str(path))
print(f"Initializer dtypes for {path.name}:\n")
for init in m.graph.initializer:
    dtype = dtype_map.get(init.data_type, str(init.data_type))
    print(f"  {init.name[:60]:<60} {dtype}")


Initializer dtypes for resnet_42_int4_qdq.onnx:

  conv1.weight                                                 FLOAT
  conv1.weight_bias                                            FLOAT
  fc.weight_i4                                                 INT4
  fc.weight_scale                                              FLOAT
  val_230                                                      INT64
  layer1.0.conv1.weight                                        FLOAT
  layer1.0.conv1.weight_bias                                   FLOAT
  layer1.0.conv2.weight                                        FLOAT
  layer1.0.conv2.weight_bias                                   FLOAT
  layer1.1.conv1.weight                                        FLOAT
  layer1.1.conv1.weight_bias                                   FLOAT
  layer1.1.conv2.weight                                        FLOAT
  layer1.1.conv2.weight_bias                                   FLOAT
  layer2.0.conv1.weight                                

## Notes

In [26]:
# =============================================================================
# Q/DQ ONNX Export — Notes
# =============================================================================
#
# INT4 — 0 Q, 1 DQ (weight-only quantization)
#   INT4 has only 16 discrete values — far too coarse for activations.
#   Weights are pre-quantized to INT4 statically at export time and stored
#   as INT4 constants in the graph. No Q node is needed at runtime.
#   Only a DQ is inserted to convert INT4 weights -> float before the
#   compute op. Activations stay in float entirely.
#   Graph pattern: INT4 weight (constant) -> DQ -> MatMul(activation, dequantized_weight)
#   The "1 DQ" means only 1 layer was eligible (INT4 is too coarse for
#   most of ResNet18's small layers).
#
# INT8 — 41 Q, 41 DQ (full activation + weight quantization)
#   Both weights and activations are quantized dynamically.
#   Every quantizable op gets a Q/DQ pair on inputs:
#   ~20 Conv layers x 2 tensors (weight + activation) ~ 40, plus the
#   final FC layer = 41.
#
# FP8 — 38 Q, 38 DQ (3 fewer than INT8)
#   Same scheme as INT8 (dynamic Q/DQ for both weights and activations),
#   but 3 fewer pairs. ModelOpt's FP8 mode excludes certain layers:
#   - First Conv layer: input pixel distributions don't fit FP8's narrow range
#   - Residual Add ops: FP8 arithmetic isn't supported for them in TensorRT
#   - Final FC/classifier: excluded for accuracy reasons
#   FP8 format (E4M3 or E5M2) has a much smaller representable range than INT8.
#
# ResNet18 Q/DQ counts:
#   ResNet18 has ~20 Conv layers. Each gets one Q/DQ for weights and one
#   for activations, giving ~40 pairs. Small differences reflect which
#   layers each mode's calibration decided to quantize.
#
# The naive approach (cell 6 in original notebook) of passing
# op_types_to_quantize=["Conv","MatMul","Gemm","Add"] doesn't work for FP8.
# FP8 requires explicit nodes_to_quantize — we load the base ONNX and
# enumerate all nodes whose op_type is in {Conv, Gemm, MatMul, Add}.
# =============================================================================